#  VBB GTFS Realtime

The VBB GTFS Realtime feed can be found at the following URL:   
https://production.gtfsrt.vbb.de/data

You can inspect the feed in a web-based GTFS-RT inspector.   
It is licensed as CC-BY 4.0.   
You can fetch it without authentication, up to a limit of 60 requests per minute.   
Please make sure to subscribe to the Atom (RSS) feed to get notified about downtime, important changes, etc.   
It has CORS enabled, so you can query it from any webpage.   
It is served with an ETag header, allowing clients to cache it.   
The code behind this is open.   

In [1]:
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd

# URL des VBB GTFS-RT Feeds
URL = "https://production.gtfsrt.vbb.de/data"

def get_vbb_delays():
    # 1. Feed herunterladen
    response = requests.get(URL)
    
    # 2. Protobuf-Parser initialisieren
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)
    
    delay_data = []
    
    # 3. Durch die Entities loopen (TripUpdates enthalten die Verspätungen)
    for entity in feed.entity:
        if entity.HasField('trip_update'):
            trip_id = entity.trip_update.trip.trip_id
            route_id = entity.trip_update.trip.route_id
            
            # Wir nehmen die Verspätung des nächsten/aktuellen Halts
            for stop_time_update in entity.trip_update.stop_time_update:
                delay = stop_time_update.arrival.delay # Verspätung in Sekunden
                stop_id = stop_time_update.stop_id
                
                delay_data.append({
                    'trip_id': trip_id,
                    'route_id': route_id,
                    'stop_id': stop_id,
                    'delay_sec': delay
                })
    
    return pd.DataFrame(delay_data)

# Ersten Blick in die Live-Daten werfen
df_live_delays = get_vbb_delays()
print(f"Aktuell {len(df_live_delays)} Verspätungsmeldungen empfangen.")
display(df_live_delays.head())

Aktuell 937051 Verspätungsmeldungen empfangen.


,trip_id,route_id,stop_id,delay_sec
0,292069215,27278_700,de:12067:900310001::5,0
1,292069215,27278_700,de:12067:900311149::1,0
2,292069215,27278_700,de:12067:900310835::1,0
3,292069215,27278_700,de:12067:900310825::1,0
4,292069215,27278_700,de:12067:900310805::1,0
